# Chapitre 10 — Optimiser latence, coûts et scalabilité

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-10-optimisation/10_optimisation.ipynb)

Ce notebook transforme les six extraits du chapitre en expériences exécutables. OpenAI n'est utilisé que pour le streaming, le résumé et le routage ; Hugging Face est facultatif pour le token pruning.

## Ressources utiles

- [OpenAI Docs — Prompt Caching](https://developers.openai.com/api/docs/guides/prompt-caching)
- [OpenAI Docs — Batch API](https://developers.openai.com/api/docs/guides/batch)
- [OpenAI Docs — optimisation de la latence](https://developers.openai.com/api/docs/guides/latency-optimization)
- [Hugging Face — pipelines Transformers](https://huggingface.co/docs/transformers/main_classes/pipelines)

## 0. Préparer Colab ou Jupyter

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[app,optimization]"],
    check=True,
)
print("Environnement du chapitre 10 prêt :", Path.cwd())


## Modèles facultatifs

Laissez les deux cases décochées pour exécuter les démonstrations locales. La clé OpenAI est saisie de manière masquée et n'est jamais enregistrée.

In [ ]:
# @title Activer uniquement les modèles que vous voulez utiliser
UTILISER_OPENAI = False # @param {type:"boolean"}
UTILISER_HUGGINGFACE = False # @param {type:"boolean"}

import os
import subprocess
import sys
from getpass import getpass

if UTILISER_OPENAI:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"], check=True)
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    os.environ.setdefault("OPENAI_SMALL_MODEL", os.environ["OPENAI_MODEL"])
    os.environ.setdefault("OPENAI_LARGE_MODEL", os.environ["OPENAI_MODEL"])

if UTILISER_HUGGINGFACE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[huggingface]"],
        check=True,
    )

print("OpenAI :", "actif" if UTILISER_OPENAI else "inactif")
print("Hugging Face :", "actif" if UTILISER_HUGGINGFACE else "inactif")


## 1. Streaming SSE

Réduire la latence perçue en envoyant les deltas, puis les sources et la fin.

Script correspondant : [`01_streaming.py`](examples/01_streaming.py)

In [ ]:
# ruff: noqa: F811
"""Diffusion SSE d'une réponse OpenAI avec sources en dernier événement."""

from __future__ import annotations

import json
import os
from collections.abc import Iterable, Iterator, Sequence
from dataclasses import dataclass


@dataclass(frozen=True)
class Passage:
    texte: str
    source: str
    page: int = 0


def _client(client=None):
    if client is not None:
        return client
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
    from openai import OpenAI

    return OpenAI()


def evenements_sse(
    question: str,
    passages: Sequence[Passage],
    *,
    model: str | None = None,
    client=None,
) -> Iterator[str]:
    """Émet les deltas de texte, les sources, puis le marqueur de fin."""

    selected_model = model or os.getenv("OPENAI_MODEL")
    if not selected_model:
        raise RuntimeError("Configurez OPENAI_MODEL avant d'activer le streaming")
    contexte = "\n\n".join(
        f"[doc_{index}] {passage.source}, page {passage.page}\n{passage.texte}"
        for index, passage in enumerate(passages, start=1)
    )
    stream: Iterable[object] = _client(client).responses.create(
        model=selected_model,
        instructions=(
            "Réponds uniquement à partir des extraits. "
            "Cite chaque affirmation avec [doc_N]."
        ),
        input=f"EXTRAITS\n{contexte}\n\nQUESTION\n{question}",
        stream=True,
    )
    for event in stream:
        if getattr(event, "type", "") == "response.output_text.delta":
            delta = getattr(event, "delta", "")
            if delta:
                yield f"data: {json.dumps({'delta': delta}, ensure_ascii=False)}\n\n"

    sources = [
        {"source": passage.source, "page": passage.page}
        for passage in passages
    ]
    yield f"data: {json.dumps({'sources': sources}, ensure_ascii=False)}\n\n"
    yield "data: [FIN]\n\n"


def reponse_streaming(question: str, retriever, **kwargs: object):
    """Adaptateur FastAPI facultatif, importé seulement si nécessaire."""

    from fastapi.responses import StreamingResponse

    passages = retriever.chercher(question)
    return StreamingResponse(
        evenements_sse(question, passages, **kwargs),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )


if __name__ == "__main__":
    print("Exemple prêt : configurez OPENAI_API_KEY et OPENAI_MODEL, puis appelez evenements_sse().")


## 2. Cache sémantique

Réutiliser une réponse proche avec TTL, éviction LRU et taux de succès.

Script correspondant : [`02_cache_semantique.py`](examples/02_cache_semantique.py)

In [ ]:
# ruff: noqa: F811
"""Cache sémantique avec expiration, éviction LRU et statistiques."""

from __future__ import annotations

import time
from collections.abc import Callable
from dataclasses import dataclass

import numpy as np


@dataclass
class Entree:
    vecteur: np.ndarray
    reponse: object
    cree_a: float
    dernier_acces: float
    nb_acces: int = 0


class CacheSemantique:
    def __init__(
        self,
        encodeur,
        *,
        seuil: float = 0.95,
        duree_vie: float = 3_600,
        taille_max: int = 1_000,
        horloge: Callable[[], float] = time.time,
    ) -> None:
        if not 0 <= seuil <= 1:
            raise ValueError("seuil doit être compris entre 0 et 1")
        if duree_vie <= 0 or taille_max <= 0:
            raise ValueError("duree_vie et taille_max doivent être strictement positifs")
        self.encodeur = encodeur
        self.seuil = seuil
        self.duree_vie = duree_vie
        self.taille_max = taille_max
        self.horloge = horloge
        self.entrees: dict[str, Entree] = {}
        self.stats = {"succes": 0, "echecs": 0}

    def chercher(self, question: str):
        self._purger()
        if not self.entrees:
            self.stats["echecs"] += 1
            return None

        vecteur = self._encoder(question)
        cles = list(self.entrees)
        matrice = np.vstack([self.entrees[cle].vecteur for cle in cles])
        similarites = matrice @ vecteur
        meilleur = int(np.argmax(similarites))
        if float(similarites[meilleur]) < self.seuil:
            self.stats["echecs"] += 1
            return None

        entree = self.entrees[cles[meilleur]]
        entree.dernier_acces = self.horloge()
        entree.nb_acces += 1
        self.stats["succes"] += 1
        return entree.reponse

    def enregistrer(self, question: str, reponse: object) -> None:
        self._purger()
        if question not in self.entrees and len(self.entrees) >= self.taille_max:
            self._evincer()
        maintenant = self.horloge()
        self.entrees[question] = Entree(
            vecteur=self._encoder(question),
            reponse=reponse,
            cree_a=maintenant,
            dernier_acces=maintenant,
        )

    def taux_de_succes(self) -> float:
        total = self.stats["succes"] + self.stats["echecs"]
        return self.stats["succes"] / total if total else 0.0

    def _encoder(self, texte: str) -> np.ndarray:
        if hasattr(self.encodeur, "embed"):
            brut = self.encodeur.embed([texte])[0]
        else:
            brut = self.encodeur.encode(texte)
        vecteur = np.asarray(brut, dtype=float)
        norme = np.linalg.norm(vecteur)
        if not norme:
            raise ValueError("L'encodeur a produit un vecteur nul")
        return vecteur / norme

    def _purger(self) -> None:
        limite = self.horloge() - self.duree_vie
        for cle in [cle for cle, entree in self.entrees.items() if entree.cree_a < limite]:
            del self.entrees[cle]

    def _evincer(self) -> None:
        cle = min(self.entrees, key=lambda item: self.entrees[item].dernier_acces)
        del self.entrees[cle]


if __name__ == "__main__":
    class EncodeurDemo:
        def encode(self, texte: str) -> list[float]:
            return [float("retour" in texte.lower()), 0.1]

    cache = CacheSemantique(EncodeurDemo(), seuil=0.9)
    cache.enregistrer("Quel est le délai de retour ?", "30 jours")
    print(cache.chercher("Quel est le délai de retour ?"), cache.taux_de_succes())


## 3. Token pruning

Conserver uniquement les phrases pertinentes avec un classifieur Hugging Face.

Script correspondant : [`03_token_pruning.py`](examples/03_token_pruning.py)

In [ ]:
# ruff: noqa: F811
"""Réduction du contexte par score de pertinence phrase-question."""

from __future__ import annotations

import re
from collections.abc import Callable

SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+")


class TokenPruner:
    def __init__(
        self,
        threshold: float = 0.3,
        *,
        scorer: Callable[[str, str], float] | None = None,
        model: str = "facebook/bart-large-mnli",
    ) -> None:
        if not 0 <= threshold <= 1:
            raise ValueError("threshold doit être compris entre 0 et 1")
        self.threshold = threshold
        self.scorer = scorer or self._huggingface_scorer(model)

    @staticmethod
    def _huggingface_scorer(model: str) -> Callable[[str, str], float]:
        try:
            from transformers import pipeline
        except ImportError as exc:
            raise RuntimeError('Installez les modèles avec pip install -e ".[huggingface]"') from exc
        classifier = pipeline("zero-shot-classification", model=model)

        def score(sentence: str, question: str) -> float:
            result = classifier(sentence, [question], multi_label=True)
            return float(result["scores"][0])

        return score

    def prune_context(self, context: str, question: str) -> str:
        sentences = [item.strip() for item in SENTENCE_BOUNDARY.split(context) if item.strip()]
        retained = [
            sentence
            for sentence in sentences
            if self.scorer(sentence, question) >= self.threshold
        ]
        return " ".join(retained)


if __name__ == "__main__":
    def score_demo(sentence: str, question: str) -> float:
        words = set(question.lower().split())
        return len(words & set(sentence.lower().split())) / max(len(words), 1)

    pruner = TokenPruner(0.1, scorer=score_demo)
    print(pruner.prune_context("Retours sous 30 jours. Livraison gratuite.", "délai retours"))


## 4. Résumé sélectif

Résumer seulement les chunks qui dépassent le budget de contexte.

Script correspondant : [`04_summarize_chunks.py`](examples/04_summarize_chunks.py)

In [ ]:
# ruff: noqa: F811
"""Résumé sélectif des chunks longs avec un modèle OpenAI léger."""

from __future__ import annotations

import os
from collections.abc import Callable, Sequence


def compter_tokens_approximatifs(text: str) -> int:
    return max(1, (len(text) + 3) // 4)


def resumer_chunks(
    chunks: Sequence[str],
    resumer: Callable[[str], str],
    *,
    seuil_tokens: int = 500,
    mots_max: int = 100,
) -> list[str]:
    """Ne paie un résumé que pour les chunks qui dépassent le seuil."""

    if seuil_tokens <= 0 or mots_max <= 0:
        raise ValueError("Les seuils doivent être strictement positifs")
    outputs = []
    for chunk in chunks:
        if compter_tokens_approximatifs(chunk) <= seuil_tokens:
            outputs.append(chunk)
            continue
        prompt = (
            f"Résume le texte suivant en {mots_max} mots maximum. "
            "Conserve les faits, chiffres, conditions et exceptions.\n\n"
            f"{chunk}"
        )
        outputs.append(resumer(prompt))
    return outputs


def resumer_avec_openai(*, client=None, model: str | None = None) -> Callable[[str], str]:
    selected_model = model or os.getenv("OPENAI_SMALL_MODEL") or os.getenv("OPENAI_MODEL")
    if not selected_model:
        raise RuntimeError("Configurez OPENAI_SMALL_MODEL ou OPENAI_MODEL")
    if client is None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        from openai import OpenAI

        client = OpenAI()

    def summarize(prompt: str) -> str:
        response = client.responses.create(model=selected_model, input=prompt)
        return str(response.output_text)

    return summarize


if __name__ == "__main__":
    print("Exemple prêt : injectez resumer_avec_openai() dans resumer_chunks().")


## 5. Routage économique

Choisir un modèle selon la complexité et la criticité de la question.

Script correspondant : [`05_routeur_economique.py`](examples/05_routeur_economique.py)

In [ ]:
# ruff: noqa: F811
"""Routage économique en deux étages avec génération OpenAI."""

from __future__ import annotations

import os
from collections.abc import Callable
from enum import Enum


class Complexite(Enum):
    SIMPLE = "simple"
    MOYENNE = "moyenne"
    COMPLEXE = "complexe"


SUJETS_CRITIQUES = (
    "juridique",
    "medical",
    "médical",
    "contrat",
    "sanction",
    "licenciement",
    "conformite",
    "conformité",
)
MARQUEURS_SIMPLES = ("quel est", "combien", "quand", "qui est", "quel montant")
MARQUEURS_COMPLEXES = (
    "compare",
    "implique",
    "synthèse",
    "synthese",
    "analyse",
    "impact de",
    "différence entre",
    "difference entre",
    "pourquoi",
)


def classer(
    question: str,
    classifieur: Callable[[str], str] | None = None,
) -> Complexite:
    texte = question.casefold()
    if any(sujet in texte for sujet in SUJETS_CRITIQUES):
        return Complexite.COMPLEXE
    if any(marker in texte for marker in MARQUEURS_COMPLEXES):
        return Complexite.COMPLEXE
    if any(marker in texte for marker in MARQUEURS_SIMPLES) and len(texte.split()) < 15:
        return Complexite.SIMPLE
    if classifieur is None:
        return Complexite.MOYENNE
    try:
        return Complexite(classifieur(question).strip().lower())
    except ValueError:
        return Complexite.MOYENNE


def modele_pour(complexite: Complexite) -> str:
    small = os.getenv("OPENAI_SMALL_MODEL") or os.getenv("OPENAI_MODEL")
    large = os.getenv("OPENAI_LARGE_MODEL") or os.getenv("OPENAI_MODEL")
    selected = large if complexite is Complexite.COMPLEXE else small
    if not selected:
        raise RuntimeError("Configurez OPENAI_MODEL ou les variantes SMALL/LARGE")
    return selected


def repondre(
    question: str,
    *,
    instructions: str,
    classifieur: Callable[[str], str] | None = None,
    client=None,
) -> tuple[str, Complexite, str]:
    complexity = classer(question, classifieur)
    model = modele_pour(complexity)
    if client is None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        from openai import OpenAI

        client = OpenAI()
    response = client.responses.create(model=model, instructions=instructions, input=question)
    return str(response.output_text), complexity, model


if __name__ == "__main__":
    for example in ("Quel est le délai ?", "Compare les trois politiques", "Explique la règle"):
        print(example, "->", classer(example).value)


## 6. Embeddings par lots

Préserver l'ordre, reprendre après erreur temporaire et suivre la progression.

Script correspondant : [`06_batch_embedding.py`](examples/06_batch_embedding.py)

In [ ]:
# ruff: noqa: F811
"""Vectorisation par lots avec reprise exponentielle et progression."""

from __future__ import annotations

import logging
import time
from collections.abc import Callable, Sequence

logger = logging.getLogger(__name__)


def _encoder(encodeur, textes: Sequence[str]) -> list[list[float]]:
    if hasattr(encodeur, "embed"):
        return encodeur.embed(textes)
    return encodeur.encode(list(textes))


def vectoriser_lot(
    textes: Sequence[str],
    encodeur,
    *,
    tentatives: int = 5,
    attente_initiale: float = 2.0,
    exceptions_reessayables: tuple[type[Exception], ...] = (TimeoutError, ConnectionError),
    dormir: Callable[[float], None] = time.sleep,
) -> list[list[float]]:
    for tentative in range(tentatives):
        try:
            return _encoder(encodeur, textes)
        except exceptions_reessayables:
            if tentative == tentatives - 1:
                raise
            dormir(attente_initiale * (2**tentative))
    raise RuntimeError("Boucle de reprise incohérente")


def vectoriser_corpus(
    chunks: Sequence[str],
    encodeur,
    *,
    taille_lot: int = 512,
    pause: float = 0.1,
    dormir: Callable[[float], None] = time.sleep,
) -> list[list[float]]:
    if taille_lot <= 0 or pause < 0:
        raise ValueError("taille_lot doit être positif et pause ne peut pas être négative")
    if not chunks:
        return []

    vectors: list[list[float]] = []
    batch_count = (len(chunks) + taille_lot - 1) // taille_lot
    started = time.perf_counter()
    for number in range(batch_count):
        batch = chunks[number * taille_lot : (number + 1) * taille_lot]
        vectors.extend(vectoriser_lot(batch, encodeur, dormir=dormir))
        if pause and number < batch_count - 1:
            dormir(pause)
        elapsed = time.perf_counter() - started
        remaining = elapsed / len(vectors) * (len(chunks) - len(vectors))
        logger.info(
            "%d/%d chunks (%.0f %%) — reste ~%.0f s",
            len(vectors),
            len(chunks),
            100 * len(vectors) / len(chunks),
            remaining,
        )
    return vectors


if __name__ == "__main__":
    class EncodeurDemo:
        def embed(self, textes: Sequence[str]) -> list[list[float]]:
            return [[float(len(texte)), 1.0] for texte in textes]

    print(vectoriser_corpus(["un", "deux", "trois"], EncodeurDemo(), taille_lot=2, pause=0))


## Exécuter les six démonstrations locales

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "chapters/chapitre-10-optimisation/runnable/run_chapter.py"],
    check=True,
)


## Bilan

Optimisez dans cet ordre : mesure, latence perçue, suppression du travail inutile, réduction du contexte, routage des modèles, puis parallélisation et mise à l'échelle. Chaque optimisation doit conserver les métriques de qualité du chapitre 7.